In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================
# Update these paths to match your computer
base_dir = r'C:\Users\marir\Pythonscripts'
nc_folder = os.path.join(base_dir, 'era5_data_2025')
output_folder = os.path.join(base_dir, 'global_cities_climate_data')
# This should be the file with columns 'Country' and 'Representative City'
master_file = os.path.join(base_dir, '53 countries lat and long .xlsx')

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

print("=" * 70)
print("ERA5 NetCDF to Excel Converter - Multi-Year Incremental Processing")
print("=" * 70)

# Step 1: Read master spreadsheet for city-country mapping
print("[1/4] Reading master spreadsheet...")
try:
    master_df = pd.read_excel(master_file, sheet_name='Countries_cities', engine='openpyxl')
    master_df['Country'] = master_df['Country'].ffill()
    
    # Create mapping: {City: Country}
    city_country_map = {}
    for _, row in master_df.iterrows():
        city_raw = str(row['Representative City']).split(',')[0].strip()
        country_raw = str(row['Country']).replace(' ', '_').strip()
        city_country_map[city_raw] = country_raw
    
    print(f"  ✓ Created mapping for {len(city_country_map)} cities")
    
except Exception as e:
    print(f"  ✗ Error reading master file: {e}")
    raise

# Step 2: Scan and group .nc files by City and Year
print("[2/4] Scanning NetCDF files...")
nc_files = glob.glob(os.path.join(nc_folder, '*.nc'))
print(f"  Found {len(nc_files)} total .nc files")

# Structure: {city: {year: [list_of_12_nc_files]}}
city_year_files = {}

for nc_file in nc_files:
    filename = os.path.basename(nc_file)
    base_filename = filename.replace('.nc', '')
    
    parts = base_filename.split('_')
    if len(parts) >= 4:
        try:
            year = int(parts[-2])
            month = int(parts[-1])
            # Reconstruct city name (first part)
            potential_city = parts[0].strip()
            
            if potential_city in city_country_map:
                if potential_city not in city_year_files:
                    city_year_files[potential_city] = {}
                if year not in city_year_files[potential_city]:
                    city_year_files[potential_city][year] = []
                
                city_year_files[potential_city][year].append(nc_file)
        except ValueError:
            continue

# Step 3: Mapping of ERA5 short names to readable names
rename_map = {
    't2m': '2m_Temperature_C',
    'skt': 'Skin_Temperature_C',
    'd2m': '2m_Dewpoint_Temperature_C',
    'tp': 'Total_Precipitation_mm',
    'sf': 'Snowfall_mm',
    'sd': 'Snow_Depth_mm',
    'rsn': 'Snow_Density_kg_m3',
    'uvb': 'UV_Radiation_J_m2',
    'tcc': 'Total_Cloud_Cover_0-1',
    'ssrd': 'Surface_Solar_Radiation_Downwards_J_m2',
    'u10': '10m_U_Wind_ms',
    'v10': '10m_V_Wind_ms',
    'swh': 'Wave_Height_m',
    'mwp': 'Mean_Wave_Period_s',
    'lsm': 'Land_Sea_Mask'
}

# Step 4: Process cities incrementally
print("[3/4] Processing cities incrementally...")
processed_count = 0
updated_count = 0

for city_name, year_dict in sorted(city_year_files.items()):
    country_name = city_country_map[city_name]
    # Updated output name to follow cityname_countryname.xlsx format
    output_filename = f"{city_name}_{country_name}.xlsx"
    output_path = os.path.join(output_folder, output_filename)
    
    existing_df = pd.DataFrame()
    existing_years = set()
    
    # Load existing data if file exists
    if os.path.exists(output_path):
        try:
            existing_df = pd.read_excel(output_path, engine='openpyxl')
            if 'Year' in existing_df.columns:
                existing_years = set(existing_df['Year'].unique())
        except Exception as e:
            print(f"  ⚠ Error reading existing file {output_filename}: {e}")

    # Identify new years that have 12 months of data and are not in the Excel yet
    new_years_to_process = []
    for year, files in year_dict.items():
        if len(files) == 12 and year not in existing_years:
            new_years_to_process.append(year)
    
    if not new_years_to_process:
        continue

    print(f"  → Processing {city_name}: Adding years {sorted(new_years_to_process)}")
    
    try:
        new_data_dfs = []
        for year in sorted(new_years_to_process):
            for nc_file in sorted(year_dict[year]):
                with xr.open_dataset(nc_file) as ds:
                    df = ds.to_dataframe().reset_index()
                    new_data_dfs.append(df)
        
        combined_new_df = pd.concat(new_data_dfs, ignore_index=True)
        
        # --- Data Transformations ---
        combined_new_df['datetime'] = pd.to_datetime(combined_new_df['valid_time'])
        combined_new_df['Year'] = combined_new_df['datetime'].dt.year
        combined_new_df['Month'] = combined_new_df['datetime'].dt.month
        combined_new_df['Day'] = combined_new_df['datetime'].dt.day
        combined_new_df['Hour'] = combined_new_df['datetime'].dt.hour
        
        # Conversions (Kelvin to C, meters to mm)
        for c in ['t2m', 'skt', 'd2m']:
            if c in combined_new_df.columns: combined_new_df[c] -= 273.15
        for c in ['tp', 'sf', 'sd']:
            if c in combined_new_df.columns: combined_new_df[c] *= 1000
        if 'u10' in combined_new_df.columns and 'v10' in combined_new_df.columns:
            combined_new_df['Resultant_Wind_Speed_ms'] = np.sqrt(combined_new_df['u10']**2 + combined_new_df['v10']**2)

        combined_new_df = combined_new_df.rename(columns=rename_map)
        
        # Column selection
        final_cols = ['Year', 'Month', 'Day', 'Hour'] + [v for v in rename_map.values() if v in combined_new_df.columns]
        if 'Resultant_Wind_Speed_ms' in combined_new_df.columns:
            final_cols.append('Resultant_Wind_Speed_ms')
            
        filtered_new_df = combined_new_df[[c for c in final_cols if c in combined_new_df.columns]]
        
        # Merge with existing
        final_output_df = pd.concat([existing_df, filtered_new_df], ignore_index=True)
        final_output_df = final_output_df.sort_values(['Year', 'Month', 'Day', 'Hour']).drop_duplicates()
        
        # Save
        final_output_df.to_excel(output_path, index=False, engine='openpyxl')
        
        if not existing_df.empty: 
            updated_count += 1
        else: 
            processed_count += 1
        print(f"    ✓ Successfully updated {output_filename}")
        
    except Exception as e:
        print(f"    ✗ Error processing {city_name}: {e}")

print("\n" + "=" * 70)
print(f"FINISHED: {processed_count} new files created, {updated_count} files updated.")
print(f"Output folder: {output_folder}")
print("=" * 70)